<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_02_graph_basics.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 02 — Graphs from Scratch

**Deep Learning for Engineering · Aalborg University · Part 1**

Notebook 01 applied a learned local rule on a grid. A grid is a very particular
graph: every pixel has the same four neighbours, in the same directions, at the
same distance. Take those regularities away and the convolution stops making
sense — but the *idea* survives, and what survives is a graph neural network.

This notebook builds every piece of that idea in NumPy before any library is
allowed near it. By the end you will have written message passing yourself, in
about five lines, and seen that a graph convolution layer is one matrix product
away from an ordinary dense layer.

There is no training here and no `torch`. That is deliberate. A graph network is
usually met as a library call, and students then carry a vague picture of
"something with neighbours" into a problem where the details decide whether it
works. The details are all in this notebook.

The graph is the **six-bus power network** you will train on in notebook 03, and
which **Ex_12.1 in Part 2** works on as well.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import Ex_5_core as core

np.set_printoptions(precision=3, suppress=True)

edges = core.six_bus_edges()
print("buses:", len(core.BUS_NAMES))
for i, name in enumerate(core.BUS_NAMES):
    print(f"  {i}  {name}")
print("\nlines:")
for a, b, susc in core.SIX_BUS_LINES:
    print(f"  {a} -- {b}   susceptance {susc:.1f} p.u.")

core.plot_graph(title="The six-bus network")
plt.show()

**What you should see.** Six named buses, eight lines with their
susceptances, and the network drawn with bus 0 on the left and bus 5 on the
right.

Take a moment over the picture, because notebook 03 assumes you know it. Buses 0
and 1 are generation. Buses 2, 3 and 4 are load centres. Bus 5 is an HVDC link
arriving from a neighbouring system, which appears to this network as a fixed
injection — it is not a machine and it does not respond to frequency. The
topology is loosely in the spirit of a Danish transmission corridor: generation
at one end, load in the middle, an interconnector at the far end.

The line parameters are invented. Say so whenever you quote a result from this
exercise set.

---

## 1 · The adjacency matrix

A graph is a set of nodes and a set of edges. The adjacency matrix is the
bookkeeping: $A_{ij} = 1$ if there is an edge from $i$ to $j$, and 0 otherwise.

### Your turn

Build it from the edge list. Do not call `core.six_bus_adjacency()` — write it,
then compare against it.

In [ ]:
# TODO: build the 6 x 6 adjacency matrix from the edge list `edges`.
#
#   A = np.zeros((6, 6))
#   for (a, b) in edges:
#       ... set both A[a, b] and A[b, a], because a line carries power both ways
#
# Then check it against core.six_bus_adjacency().

raise NotImplementedError("Build the adjacency matrix A from the edge list")

In [ ]:
print("A =")
print(A.astype(int))
print("\nmatches core.six_bus_adjacency():", np.array_equal(A, core.six_bus_adjacency()))
print("symmetric      :", np.array_equal(A, A.T))
print("zero diagonal  :", np.all(np.diag(A) == 0))
print("edges counted  :", int(A.sum() // 2), "  (should be", len(edges), ")")
print("degrees        :", A.sum(axis=1).astype(int))

core.show_matrix(A, title="Adjacency matrix", fmt="{:.0f}", labels=core.BUS_NAMES)
plt.show()

**What you should see.** `matches ... True`, `symmetric : True`, `zero
diagonal : True`, `edges counted : 8`, and degrees `[2 3 3 3 3 2]`.

Three properties, three modelling decisions hiding inside them.

**Symmetric** because a transmission line is bidirectional. A graph of a process
plant with one-way flows would not be, and then `A[i, j]` and `A[j, i]` mean
different things and every formula below needs care about which index is which.

**Zero diagonal** because a bus is not its own neighbour. You will add a self
loop deliberately in section 4, and the reason you have to add it deliberately is
that it was not there to begin with.

**Unweighted** — every entry is 0 or 1, so the adjacency matrix says a line
exists but not how strong it is. The susceptances are thrown away. That is a
loss of information and a real modelling choice: `core.susceptance_matrix()`
keeps them, and a more careful graph network would use them as **edge features**.
L12.2 in Part 2 does exactly that, feeding line conductance and susceptance in as
edge features. Ex_05 does not, so that the arithmetic stays visible.

---

## 2 · Neighbourhoods and reach

The neighbourhood $\mathcal{N}(i)$ of node $i$ is the set of nodes joined to it
by an edge. It is the only thing a message-passing layer is allowed to look at.

### Your turn

Write `neighbours(A, i)` and use it to list the neighbourhood of every bus. Then
answer the question that decides how deep the network in notebook 03 has to be:
**how many hops does it take to get from any bus to any other?**

Powers of the adjacency matrix answer that. $(A^k)_{ij}$ counts the walks of
length exactly $k$ from $i$ to $j$; so the smallest $k$ for which
$(I + A + A^2 + \dots + A^k)_{ij} > 0$ is the distance between $i$ and $j$.

In [ ]:
# TODO: (a) list the neighbours of each bus, and
#       (b) find the smallest number of hops that connects every pair.
#
#   def neighbours(A, i):
#       return np.nonzero(A[i])[0]
#
#   reach = np.eye(6)
#   for k in 1, 2, 3, ...:
#       reach = reach + matrix power of A up to k
#       if every entry of reach is > 0: the diameter is k, stop
#
# Record the answer in `diameter`.

raise NotImplementedError("Write neighbours(A, i) and find the graph diameter")

In [ ]:
for i in range(core.N_BUS):
    nb = neighbours(A, i)
    print(f"  bus {i} ({core.BUS_NAMES[i]:8s}) neighbours: {list(nb)}")

print("\nA @ A (walks of length two):")
print((A @ A).astype(int))
print("\ndiameter:", diameter, "hops")

**What you should see.** Neighbour lists

```
bus 0 neighbours: [1, 2]
bus 1 neighbours: [0, 2, 3]
bus 2 neighbours: [0, 1, 4]
bus 3 neighbours: [1, 4, 5]
bus 4 neighbours: [2, 3, 5]
bus 5 neighbours: [3, 4]
```

the matrix $A^2$, and `diameter: 3 hops`.

The diagonal of $A^2$ is the degree of each node — a walk of length two that
returns to where it started must go out along an edge and back along the same
one. The off-diagonal entries count common neighbours: $(A^2)_{03} = 2$ says
buses 0 and 3 are joined by two different two-hop routes, through bus 1 and
through bus 2.

**The diameter is the number to remember.** It is three, so a node needs at least
three message-passing layers before it can be influenced by the farthest bus at
all. A one-layer graph network on this system is structurally incapable of
letting bus 0 know anything about bus 5, no matter how long you train it or how
wide you make it. Notebook 03 measures exactly this, and L12.2 in Part 2 says the
same thing about depth: *too few layers and a machine cannot feel a fault several
buses away within one forward pass.*

Depth in a graph network is a statement about physics, not a hyperparameter you
tune blindly.

---

## 3 · Node features, and one round of message passing

Each node carries a feature vector. Stack them and you have $H$, of shape
(nodes, features). For now, give each bus a single feature so that the
arithmetic is checkable by hand: bus 0 gets a 1 and everybody else gets a 0.

The general message-passing rule from L5.1 is

$$\mathbf{h}_i^{(l+1)} = \phi\!\left(\mathbf{h}_i^{(l)},\;
\sum_{j\in\mathcal{N}(i)} \psi\!\left(\mathbf{h}_i^{(l)},\mathbf{h}_j^{(l)}\right)\right)$$

which reads: *build a message from each neighbour, add the messages up, and
combine the total with your own state.* Every graph network in the literature is
this equation with particular choices of $\psi$ and $\phi$.

The simplest possible choice — $\psi$ is "send your own state", $\phi$ is "take
the mean of what arrived" — is enough to see the mechanism.

### Your turn

Implement one round of mean aggregation over neighbours, with two loops, and then
show that the same thing is one matrix product.

In [ ]:
# TODO: one round of mean aggregation, twice.
#
#   (a) with loops, into `h_loops`:
#         h_loops[i] = mean over j in neighbours(A, i) of h[j]
#
#   (b) with matrices, as a reusable function (the spreading demo below
#       calls it repeatedly):
#         def aggregate_matrix(A, h):
#             D_inv = np.diag(1.0 / A.sum(axis=1))
#             return D_inv @ A @ h
#         h_matrix = aggregate_matrix(A, h0)
#
# Compute both from `h0` below and check they agree.

h0 = np.zeros((6, 1))
h0[0, 0] = 1.0

raise NotImplementedError("Implement mean aggregation with loops and with matrices")

In [ ]:
print("h0        :", h0.ravel())
print("loops     :", h_loops.ravel())
print("matrices  :", h_matrix.ravel())
print("agree     :", np.allclose(h_loops, h_matrix))

h = h0.copy()
history = [h.ravel().copy()]
for step in range(4):
    h = aggregate_matrix(A, h)
    history.append(h.ravel().copy())

print("\nsignal starting at bus 0, spreading:")
print("  step      " + "".join(f"  bus {i}" for i in range(6)))
for s, row in enumerate(history):
    print(f"  {s:4d}      " + "".join(f"{v:7.3f}" for v in row))

fig, axes = plt.subplots(1, 4, figsize=(17.0, 3.4))
for s in range(4):
    core.plot_graph(A, node_values=history[s], ax=axes[s],
                    title=f"after {s} hops", cmap="viridis")
plt.show()

**What you should see.** `agree : True`, and a table in which the 1 that
started at bus 0 spreads outwards:

```
  step        bus 0  bus 1  bus 2  bus 3  bus 4  bus 5
     0        1.000  0.000  0.000  0.000  0.000  0.000
     1        0.000  0.333  0.333  0.000  0.000  0.000
     2        0.333  0.111  0.111  0.111  0.111  0.000
     3        0.111  0.185  0.185  0.074  0.074  0.111
     4        0.185  0.123  0.123  0.123  0.123  0.074
```

Four things are visible in that table, and each is a design problem in real graph
networks.

**Information travels one hop per layer.** After one round only buses 1 and 2 —
the neighbours of bus 0 — know anything. Bus 5, three hops away, is still exactly
zero after two rounds and first hears something at round three. That is the
diameter from section 2, appearing as a fact about the computation rather than
about the graph.

**Node 0 loses its own value immediately.** At step 1 bus 0 is zero, because the
rule as written averages the *neighbours* and forgets the node itself. That is
almost never what you want, and the standard fix — adding a self loop — is the
subject of section 4.

**The total is not conserved.** Mean aggregation is not a physical transport
process, and nothing here is a conservation law. If you need one, you must put it
in the loss, which is what Part 2 spends twelve exercises doing.

**It flattens.** Keep going and every node converges towards the same value. This
is **over-smoothing**, and it is the reason deep graph networks are hard: after
enough rounds of averaging, every node has the graph's average and nothing of its
own. Depth in a graph network buys reach and costs distinctiveness, and the two
have to be traded against each other.

---

## 4 · The graph convolution layer

L5.1 gives Kipf and Welling's layer in matrix form:

$$\mathbf{H}^{(l+1)} = \sigma\!\left(
\tilde{\mathbf{D}}^{-1/2}\tilde{\mathbf{A}}\,\tilde{\mathbf{D}}^{-1/2}
\mathbf{H}^{(l)}\mathbf{W}^{(l)}\right)$$

Unpack it right to left, because that is the order the computation happens in.

$\mathbf{H}^{(l)}\mathbf{W}^{(l)}$ is **an ordinary dense layer**, applied to
every node's feature vector with the same weights. Nothing graph-like about it:
it is the "learned rule". Weight sharing across nodes here is the exact analogue
of weight sharing across pixels in notebook 01, and it is why the parameter count
does not depend on how many buses there are.

$\tilde{\mathbf{A}} = \mathbf{A} + \mathbf{I}$ adds the **self loop** that
section 3 was missing, so a node keeps its own transformed features.

$\tilde{\mathbf{D}}^{-1/2}\tilde{\mathbf{A}}\tilde{\mathbf{D}}^{-1/2}$ is the
**symmetric normalisation**. Without it, a node with many neighbours receives a
larger sum simply for being well connected, and the scale of the features grows
or shrinks layer by layer. Dividing by degree on both sides keeps the largest
eigenvalue near one, which is what stops a deep stack from exploding.

### Your turn

Build $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ from `A`, then apply
one layer by hand.

In [ ]:
# TODO: build the normalised adjacency, then apply one graph convolution.
#
#   A_tilde = A + np.eye(6)
#   d       = A_tilde.sum(axis=1)
#   d_isqrt = 1.0 / np.sqrt(d)
#   A_hat   = A_tilde * d_isqrt[:, None] * d_isqrt[None, :]
#
# Then one layer, with a fixed weight matrix so the result is reproducible:
#
#   rng = np.random.default_rng(0)
#   W   = rng.normal(0.0, 0.5, size=(3, 4))     # 3 input features -> 4 output
#   H   = np.array([...])                        # supplied below, shape (6, 3)
#   H_next = np.tanh(A_hat @ H @ W)
#
# Check A_hat against core.normalised_adjacency(A).

raise NotImplementedError("Build A_hat and apply one graph convolution layer")

In [ ]:
print("A_hat =")
print(A_hat)
print("\nmatches core.normalised_adjacency(A):",
      np.allclose(A_hat, core.normalised_adjacency(A)))
print("symmetric      :", np.allclose(A_hat, A_hat.T))
print("row sums       :", A_hat.sum(axis=1))
print("largest eigenvalue:", np.round(np.max(np.abs(np.linalg.eigvalsh(A_hat))), 6))
print()
print("H      shape", H.shape, "  W shape", W.shape)
print("H_next shape", H_next.shape)
print("H_next =")
print(H_next)

**What you should see.** `matches ... True`, `symmetric : True`, row sums
`[0.911 1.039 1.039 1.039 1.039 0.911]`, a largest eigenvalue of exactly
`1.0`, and an `H_next` of shape (6, 4).

Two of those deserve comment.

**The row sums are not one.** Symmetric normalisation divides by
$\sqrt{d_i d_j}$, not by $d_i$, so the rows do not sum to one and the layer is
not a weighted average. The two low-degree buses, 0 and 5, come out slightly
below one and the rest slightly above. The alternative, $\tilde{D}^{-1}\tilde{A}$,
does give row sums of one but is not symmetric, and the symmetry is what makes
the spectral argument work. Both are used in practice.

**The largest eigenvalue is one.** That is not luck. It is the property the
normalisation was designed to give, and it is what makes a deep stack of these
layers neither explode nor vanish — the same concern L5.1 raises about residual
connections and normalisation more generally.

**The shape went from (6, 3) to (6, 4).** Six nodes in, six nodes out. A graph
convolution never changes the number of nodes; it changes how many numbers each
node carries. That is what makes it a *node-level* operation, and it is why the
task in notebook 03 can be node regression.

---

## 5 · Permutation: the property that makes it a graph network

Here is the property, from L5.1's slide:

$$f(\mathbf{P}\mathbf{X},\; \mathbf{P}\mathbf{A}\mathbf{P}^\top)
= \mathbf{P}\, f(\mathbf{X}, \mathbf{A})$$

In words: **relabel the nodes, and the answer is relabelled the same way.**
Nothing about the computation depends on the numbers you happened to write on
the buses.

This is worth being exact about, because two words get used interchangeably and
they are not the same thing.

* **Equivariant**: the output moves with the input. A node-level graph network is
  equivariant — that is the equation above.
* **Invariant**: the output does not move at all. A *graph*-level network — one
  that pools all the nodes into a single prediction, "will this network be
  stable, yes or no" — is invariant, because the pooling step throws the ordering
  away.

Equivariance at the node level plus a permutation-invariant pooling gives
invariance at the graph level. Notebook 03 does the node-level version, and
demonstrates it numerically.

### Your turn

Verify the identity on the pieces you have built. Take the permutation
`perm = [3, 0, 5, 2, 4, 1]`, meaning *the node now called 0 is the one that used
to be called 3*, build its permutation matrix, and check that permuting the
inputs permutes the output of one graph convolution layer.

In [ ]:
# TODO: check f(PX, P A P^T) = P f(X, A) for one graph convolution layer.
#
#   perm = [3, 0, 5, 2, 4, 1]
#   P    = core.permutation_matrix(perm)     # P[i, perm[i]] = 1
#
#   A_perm    = P @ A @ P.T
#   H_perm    = P @ H
#   A_hat_perm = core.normalised_adjacency(A_perm)
#
#   left  = np.tanh(A_hat_perm @ H_perm @ W)     # f(PX, PAP^T)
#   right = P @ H_next                            # P f(X, A)
#
# Report the largest absolute difference.

raise NotImplementedError("Verify permutation equivariance of the graph convolution")

In [ ]:
print("permutation:", perm)
print("\nP =")
print(P.astype(int))
print("\ndegrees before:", A.sum(axis=1).astype(int))
print("degrees after :", A_perm.sum(axis=1).astype(int))
print("\nlargest |f(PX, PAP^T) - P f(X, A)| =", gap)
print("equivariant to machine precision:", gap < 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.0))
core.plot_graph(A, ax=axes[0], title="original labelling",
                labels=[str(i) for i in range(6)])
core.plot_graph(A_perm, ax=axes[1], title="after relabelling",
                labels=[str(i) for i in range(6)])
plt.show()

**What you should see.** The permutation matrix, degrees `[2 3 3 3 3 2]`
becoming `[3 2 2 3 3 3]`, and

```
largest |f(PX, PAP^T) - P f(X, A)| = 0.0
equivariant to machine precision: True
```

Exactly zero, or a number around $10^{-16}$. This is not an approximation that
training improved; it is an algebraic identity that holds for **any** weights,
trained or random, because the only operations involved are a per-node linear map
and a sum over neighbours, and neither can see a node index.

Compare that with what a dense network on the flattened state vector would do.
Feed it the six buses in a different order and you have fed it a different input,
so you get a different answer. It could learn to be permutation invariant, from
data, approximately, for the orderings it happened to see — and L12.2 says
precisely this about why graph networks earn their place on a power system.
Notebook 03 measures the difference rather than asserting it.

The two pictures show why this matters practically. They are the same network.
Only the numbers written on the buses changed, and no engineering conclusion may
depend on those numbers.

---

## 6 · The parameter count, one more time

The last thing to notice before training anything: **the size of $W$ does not
depend on the number of nodes.**

`W` above is 3 by 4 — three input features, four output features — and it would
be 3 by 4 on a six-bus network, on the Danish transmission system, or on a single
isolated bus. That is weight sharing across nodes, and it is the same argument
as the 80 parameters in notebook 01.

In [ ]:
for n_nodes in (6, 60, 600):
    dense = (n_nodes * 3) * (n_nodes * 4) + n_nodes * 4
    gcn = 3 * 4 + 4
    print(f"  {n_nodes:4d} nodes:  dense layer {dense:9,d}   "
          f"graph conv {gcn:4d}   ratio {dense / gcn:9,.0f} x")

**What you should see.**

```
     6 nodes:  dense layer       456   graph conv   16   ratio        28 x
    60 nodes:  dense layer    43,440   graph conv   16   ratio     2,715 x
   600 nodes:  dense layer 4,322,400   graph conv   16   ratio   270,150 x
```

The graph convolution column is constant. That is what lets a model trained on
small cases say anything at all about a larger system — the "one model, many
grids" argument L12.1 makes, and the reason the architecture is worth the
trouble even when, as L12.1 also says, the topology is already known exactly.

---

## 7 · Before you move on

Answer these here.

1. The adjacency matrix threw away the line susceptances. Give one prediction
   task on this network where that loss would be fatal, and say how you would put
   the information back.
2. Section 3's aggregation flattened towards a common value. Name the phenomenon,
   and give a reason a practitioner might still choose four layers over two on
   this graph.
3. Permutation equivariance held for **random** weights. Why is that a stronger
   statement than "the trained model turned out to be equivariant"?
4. A convolution on an image is a special case of message passing. What is the
   graph, what are the node features, and what does weight sharing correspond
   to?

---

Continue with **`Ex05_03_gnn_six_bus_network.ipynb`**, which puts a load-flow
dataset on this graph and trains a network on it.

*Write your answers here.*

1.
2.
3.
4.